# OpenStudio Parametric Study Workflow

This notebook runs energy simulations for various building envelope renovation scenarios.


In [10]:
# Standard library
import json
import os
import subprocess
import time
from itertools import product
import re
import sys
from pathlib import Path
import configparser
import shutil
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
import csv
import sqlite3
# Add OpenStudio 3.11.0 Python bindings to path BEFORE importing
def detect_openstudio_python_path():
    env_path = os.environ.get("OPENSTUDIO_PYTHON_PATH")
    candidates = [
        env_path,
        "C:/Program Files/openstudio-3.11.0/Python",
        "/Applications/OpenStudio-3.11.0/Python",
    ]
    for candidate in candidates:
        if candidate and os.path.exists(candidate):
            return candidate
    return None
OPENSTUDIO_PYTHON_PATH = detect_openstudio_python_path()
if OPENSTUDIO_PYTHON_PATH and OPENSTUDIO_PYTHON_PATH not in sys.path:
    sys.path.insert(0, OPENSTUDIO_PYTHON_PATH)
elif not OPENSTUDIO_PYTHON_PATH:
    print("Warning: OpenStudio Python path not found; trying system package.")
import openstudio
# Read EC3 API Token from config.ini
def get_ec3_api_token():
    """Read EC3 API token from config.ini file."""
    script_dir = Path.cwd()
    repo_root = script_dir.parent.parent
    config_path = repo_root / "config.ini"
    if not config_path.exists():
        print(f"Warning: config.ini not found at {config_path}")
        return None
    config = configparser.ConfigParser()
    config.read(config_path)
    try:
        return config["EC3_API_TOKEN"]["API_TOKEN"]
    except KeyError:
        print("Warning: EC3_API_TOKEN not found in config.ini")
        return None
EC3_API_TOKEN = get_ec3_api_token()
# =========================
# HELPER FUNCTIONS
# =========================
def get_city_weather_files(city_name, base_weather_path):
    """Get the EPW and DDY file paths for a given city name."""
    city_folder_path = os.path.join(base_weather_path, city_name)
    if not os.path.exists(city_folder_path):
        print(f"Error: Folder '{city_folder_path}' does not exist")
        return None
    epw_file = None
    ddy_file = None
    for filename in os.listdir(city_folder_path):
        if filename.lower().endswith(".epw"):
            epw_file = os.path.join(city_folder_path, filename)
        elif filename.lower().endswith(".ddy"):
            ddy_file = os.path.join(city_folder_path, filename)
    return {"epw": epw_file, "ddy": ddy_file}
def generate_scenario_name(scenario_dict):
    """Generate a unique scenario name from scenario parameters."""
    parts = []
    if scenario_dict.get("is_baseline", False):
        parts.append("baseline")
    else:
        measure_count = sum([
            1 if scenario_dict.get("wall_r_value") else 0,
            1 if scenario_dict.get("roof_r_value") else 0,
            1 if scenario_dict.get("window_u_factor") else 0,
            1 if scenario_dict.get("door_option") else 0,
        ])
        if measure_count > 1:
            parts.append("all")
        if scenario_dict.get("wall_r_value"):
            parts.append(f"wall_r{scenario_dict['wall_r_value']}")
        if scenario_dict.get("roof_r_value"):
            parts.append(f"roof_r{scenario_dict['roof_r_value']}")
        if scenario_dict.get("window_u_factor"):
            parts.append(f"window_u{scenario_dict['window_u_factor']}")
        if scenario_dict.get("door_option"):
            door_abbrev = scenario_dict['door_option'].replace(' ', '_').replace('door', 'd')
            parts.append(f"door_{door_abbrev}")
    parts.append(scenario_dict["building_type"])
    parts.append(scenario_dict["city"])
    return "_".join(parts)
def summarize_renovation_details(scenario_name, wall_args=None, roof_args=None, window_args=None, door_args=None, window_upgrade_status=None):
    if scenario_name.startswith("baseline"):
        return "Baseline (no envelope renovation)"

    def fmt_value(value, unit=""):
        if value is None or value == "":
            return None
        if isinstance(value, float):
            text = f"{value:g}"
        else:
            text = str(value)
        return f"{text}{unit}" if unit else text

    def include_param(key, value):
        if value in [None, ""]:
            return False
        if "option" in str(key).lower() and str(value).strip().lower() == "none":
            return False
        return True

    details = []
    if wall_args:
        wall_r = wall_args.get("r_value")
        wall_mat = wall_args.get("insulation_material_type", "insulation material")
        details.append(f"Wall insulation improved to R-{wall_r} using {wall_mat}")
    if roof_args:
        roof_r = roof_args.get("r_value")
        roof_mat = roof_args.get("insulation_material_type", "insulation material")
        details.append(f"Roof insulation improved to R-{roof_r} using {roof_mat}")
    if window_args:
        panes = window_args.get("user_num_panes")
        infil_red = window_args.get("space_infiltration_reduction_percent")
        if window_upgrade_status == "failed_simple_glazing":
            window_main = f"Window system failed to upgrade to {panes}-pane because all window constructions are simple glazing objects; infiltration reduction set to {infil_red}%"
        elif window_upgrade_status == "requested_not_applied":
            window_main = f"Window system requested {panes}-pane upgrade but no window construction was replaced; infiltration reduction set to {infil_red}%"
        else:
            window_main = f"Window system upgraded to {panes}-pane with {infil_red}% infiltration reduction"

        window_parts = []
        for key in ["glass_option", "wf_option", "caulking_option", "film_option", "weatherstrip_option", "secondary_glazing_option"]:
            value = window_args.get(key)
            if include_param(key, value):
                window_parts.append(f"{key}={value}")

        caulking_thickness = fmt_value(window_args.get("caulking_thickness"), " m")
        if caulking_thickness:
            window_parts.append(f"caulking_thickness={caulking_thickness}")

        glass_thickness = fmt_value(window_args.get("glass_pane_thickness"), " m")
        if glass_thickness:
            window_parts.append(f"glass_pane_thickness={glass_thickness}")

        gap_thickness = fmt_value(window_args.get("gap_thickness"), " m")
        if gap_thickness:
            window_parts.append(f"gap_thickness={gap_thickness}")

        window_suffix = f"; parameters: {', '.join(window_parts)}" if window_parts else ""
        details.append(f"{window_main}{window_suffix}")
    if door_args:
        door_type = door_args.get("door_option", "door")
        infil_red = door_args.get("space_infiltration_reduction_percent")
        door_parts = []
        for key in ["door_bottom_seal_option", "door_top_side_seal_option"]:
            value = door_args.get(key)
            if include_param(key, value):
                door_parts.append(f"{key}={value}")
        door_suffix = f"; parameters: {', '.join(door_parts)}" if door_parts else ""
        details.append(f"Door system upgraded ({door_type}) with {infil_red}% infiltration reduction{door_suffix}")
    if not details:
        wall_match = re.search(r"wall_r([0-9.]+)", scenario_name)
        if wall_match:
            wall_r = wall_match.group(1)
            wall_mat = "Blown Fiberglass"
            details.append(f"Wall insulation improved to R-{wall_r} using {wall_mat}")
        roof_match = re.search(r"roof_r([0-9.]+)", scenario_name)
        if roof_match:
            roof_r = roof_match.group(1)
            roof_mat = "Blown Fiberglass"
            details.append(f"Roof insulation improved to R-{roof_r} using {roof_mat}")
        window_match = re.search(r"window_u([0-9.]+)", scenario_name)
        if window_match:
            window_u = window_match.group(1)
            panes = 2 if float(window_u) >= 0.30 else 3
            details.append(f"Window system requested {panes}-pane upgrade with 50.0% infiltration reduction")
        door_match = re.search(r"door_(.+?)_(SmallOffice|MediumOffice|LargeOffice|SmallHotel|LargeHotel|Warehouse|RetailStandalone|RetailStripmall|PrimarySchool|SecondarySchool)_", scenario_name)
        if door_match:
            door_type = door_match.group(1).replace("_", " ")
            door_type = re.sub(r"\bd\b", "door", door_type)
            details.append(f"Door system upgraded ({door_type}) with 30.0% infiltration reduction")
    if not details:
        return "Envelope renovation applied (details not parsed)"
    return "; ".join(details)
def detect_window_upgrade_status(model, requested_panes):
    """Detect whether requested pane upgrade was actually applied to model windows."""
    windows = [
        ss for ss in model.getSubSurfaces()
        if ss.subSurfaceType() in ["FixedWindow", "OperableWindow", "Skylight"]
    ]
    if not windows:
        return None

    upgraded_count = 0
    simple_glazing_count = 0

    for subsurface in windows:
        if not subsurface.construction().is_initialized():
            continue
        construction = subsurface.construction().get()
        construction_name = construction.nameString()

        if f"_New_{requested_panes}Pane_Construction" in construction_name:
            upgraded_count += 1

        is_simple = False
        if construction.to_LayeredConstruction().is_initialized():
            layered = construction.to_LayeredConstruction().get()
            for layer_index in range(layered.numLayers()):
                layer_material = layered.getLayer(layer_index)
                if layer_material.to_SimpleGlazing().is_initialized():
                    is_simple = True
                    break
        if is_simple:
            simple_glazing_count += 1

    if upgraded_count > 0:
        return "upgraded"
    if simple_glazing_count == len(windows):
        return "failed_simple_glazing"
    return "requested_not_applied"
def infer_window_args_from_scenario_name(scenario_name):
    """Infer basic window measure args from scenario name when explicit args are unavailable."""
    window_match = re.search(r"window_u([0-9.]+)", str(scenario_name))
    if not window_match:
        return None
    window_u = float(window_match.group(1))
    panes = 2 if window_u >= 0.30 else 3
    return {
        "user_num_panes": panes,
        "space_infiltration_reduction_percent": 50.0,
        "glass_option": "provide user_num_panes",
        "caulking_option": "acrylic",
        "caulking_thickness": 0.008,
        "wf_option": "none",
        "film_option": "none",
        "weatherstrip_option": "none",
        "secondary_glazing_option": "none",
        "glass_pane_thickness": 0.003,
        "gap_thickness": 0.013,
        "gwp_statistic": "median",
    }
def infer_wall_args_from_scenario_name(scenario_name):
    wall_match = re.search(r"wall_r([0-9.]+)", str(scenario_name))
    if not wall_match:
        return None
    return {
        "r_value": float(wall_match.group(1)),
        "insulation_material_type": "Blown Fiberglass",
        "insulation_material_lifetime": 30,
        "gwp_statistic": "median",
    }
def infer_roof_args_from_scenario_name(scenario_name):
    roof_match = re.search(r"roof_r([0-9.]+)", str(scenario_name))
    if not roof_match:
        return None
    return {
        "r_value": float(roof_match.group(1)),
        "insulation_material_type": "Blown Fiberglass",
        "insulation_material_lifetime": 30,
        "gwp_statistic": "median",
    }
def infer_door_args_from_scenario_name(scenario_name):
    door_match = re.search(r"door_(.+?)_(SmallOffice|MediumOffice|LargeOffice|SmallHotel|LargeHotel|Warehouse|RetailStandalone|RetailStripmall|PrimarySchool|SecondarySchool)_", str(scenario_name))
    if not door_match:
        return None
    door_type = door_match.group(1).replace("_", " ")
    door_type = re.sub(r"\bd\b", "door", door_type)
    return {
        "door_option": door_type,
        "space_infiltration_reduction_percent": 30.0,
        "door_bottom_seal_option": "automatic door bottom",
        "door_top_side_seal_option": "jamb weatherstrip",
        "gwp_statistic": "median",
    }
def run_osw(osw_dict, osw_filename, run_dir, openstudio_path, label):
    """Write an OSW and run it with the OpenStudio CLI. Returns True on success."""
    osw_path = os.path.join(run_dir, osw_filename)
    os.makedirs(run_dir, exist_ok=True)
    with open(osw_path, "w") as f:
        json.dump(osw_dict, f, indent=2)
    try:
        result = subprocess.run(
            [openstudio_path, "run", "-w", osw_filename],
            check=False,
            capture_output=True,
            text=True,
            cwd=run_dir,
            timeout=600,
        )
    except subprocess.TimeoutExpired:
        print(f"⏱️ {label}: timed out after 600s")
        return False
    except Exception as e:
        print(f"❌ {label}: subprocess error: {e}")
        return False
    out_osw_path = os.path.join(run_dir, "out.osw")
    if not os.path.exists(out_osw_path):
        print(f"❌ {label}: out.osw not found (OpenStudio may have crashed)")
        if result.stderr:
            print(f"   STDERR: {result.stderr[:500]}")
        return False
    with open(out_osw_path, "r") as f:
        out_osw = json.load(f)
    if out_osw.get("completed_status") != "Success":
        print(f"❌ {label}: OSW failed")
        run_log_path = os.path.join(run_dir, "run", "run.log")
        if os.path.exists(run_log_path):
            with open(run_log_path, "r") as log_f:
                for line in log_f:
                    if "ERROR" in line:
                        print(f"   ❌ LOG: {line.rstrip()}")
        return False
    return True
def apply_python_measure(model, measure_folder, measure_class_name, arguments_dict):
    """
    Apply a Python OpenStudio ModelMeasure directly to a model in-process.
    Returns True if successful, False otherwise.
    """
    measure_folder_str = str(measure_folder)
    try:
        if measure_folder_str not in sys.path:
            sys.path.insert(0, measure_folder_str)
        import measure as measure_module
        import importlib
        importlib.reload(measure_module)
        measure_class = getattr(measure_module, measure_class_name)
        osw = openstudio.WorkflowJSON()
        runner = openstudio.measure.OSRunner(osw)
        measure = measure_class()
        args = measure.arguments(model)
        arg_map = openstudio.measure.convertOSArgumentVectorToMap(args)
        for arg_name, arg_value in arguments_dict.items():
            if arg_name in arg_map:
                arg = arg_map[arg_name]
                arg.setValue(arg_value)
                arg_map[arg_name] = arg
        measure.run(model, runner, arg_map)
        result_value = runner.result().value().valueName()
        if result_value != "Success":
            print(f"  Measure result: {result_value}")
            for error in runner.result().errors():
                print(f"    ERROR: {error.logMessage()}")
            return False
        return True
    except Exception as e:
        print(f"  ERROR applying measure: {str(e)}")
        import traceback
        traceback.print_exc()
        return False
    finally:
        if measure_folder_str in sys.path:
            sys.path.remove(measure_folder_str)
def apply_reporting_measure(model_path, sql_file_path, measure_dir_path, label):
    """
    Apply the OperatingCostCarbonReportingMeasure (Python ReportingMeasure)
    in-process after simulation completes. The OSW runner can't execute Python
    measures, so we call it directly using the OpenStudio Python bindings.
    Returns True if successful, False otherwise.
    """
    measure_folder = os.path.join(measure_dir_path, "OperatingCostCarbonReportingMeasure")
    measure_folder_str = str(measure_folder)
    try:
        # Load model
        translator = openstudio.osversion.VersionTranslator()
        loaded = translator.loadModel(openstudio.toPath(str(model_path)))
        if not loaded.is_initialized():
            print(f"  ❌ {label}: could not load model for reporting measure")
            return False
        model = loaded.get()
        # Attach SQL file to model
        sql_file = openstudio.SqlFile(openstudio.toPath(str(sql_file_path)))
        model.setSqlFile(sql_file)
        # Set up runner with SQL file and model
        osw = openstudio.WorkflowJSON()
        runner = openstudio.measure.OSRunner(osw)
        runner.setLastEnergyPlusSqlFilePath(openstudio.toPath(str(sql_file_path)))
        runner.setLastOpenStudioModel(model)
        # Import and instantiate the measure
        if measure_folder_str not in sys.path:
            sys.path.insert(0, measure_folder_str)
        import measure as measure_module
        import importlib
        importlib.reload(measure_module)
        measure = measure_module.OperatingCostCarbonReport()
        # Arguments (none required for this measure Ã¢â‚¬â€ reads from CSV resources)
        args = measure.arguments(model)
        arg_map = openstudio.measure.convertOSArgumentVectorToMap(args)
        # Run
        measure.run(runner, arg_map)
        result_value = runner.result().value().valueName()
        if result_value != "Success":
            print(f"  ❌  {label}: reporting measure result: {result_value}")
            for error in runner.result().errors():
                print(f"    ❌ ERROR: {error.logMessage()}")
            return False
        # Save model with AdditionalProperties written by the reporting measure
        model.save(openstudio.toPath(str(model_path)), True)
        del model
        return True
    except Exception as e:
        print(f"  ❌  {label}: reporting measure error: {e}")
        import traceback
        traceback.print_exc()
        return False
    finally:
        if measure_folder_str in sys.path:
            sys.path.remove(measure_folder_str)
# =========================
# CORE: SINGLE SCENARIO CREATION/RUN
# =========================
def create_simulation(
    city,
    base_run_dir,
    measure_dir_path,
    base_weather_path,
    scenario_dict,
    overwrite_existing=False,
    building_type="SmallOffice",
    template="90.1-2013",
    climate_zone="ASHRAE 169-2013-5A",
    openstudio_path="openstudio",
):
    # --- Weather ---
    wf = get_city_weather_files(city, base_weather_path)
    if wf is None or wf["epw"] is None:
        print(f"âŒ Weather files not found for {city}")
        return None
    epw_path = os.path.abspath(wf["epw"])
    scenario_name = generate_scenario_name(scenario_dict)
    scenario_run_dir = os.path.abspath(os.path.join(base_run_dir, scenario_name))
    os.makedirs(scenario_run_dir, exist_ok=True)
    # Skip if already done
    sql_output_path = os.path.join(scenario_run_dir, "run", "eplusout.sql")
    if os.path.exists(sql_output_path) and not overwrite_existing:
        print(f"⚠️  Skipping {scenario_name} - simulation already exists")
        return scenario_name
    measure_paths = [os.path.abspath(measure_dir_path)]
    file_paths = [os.path.abspath(base_weather_path), os.path.dirname(epw_path)]
    prototype_step = {
        "measure_dir_name": "create_DOE_prototype_building",
        "name": "Create DOE Prototype Building",
        "arguments": {
            "building_type": building_type,
            "template": template,
            "climate_zone": climate_zone,
            "epw_file": "Not Applicable",
        },
    }
    # ====================================================================
    # BASELINE: OSW with prototype only Ã¢â€ â€™ E+ simulation runs automatically
    # Then apply Python reporting measure in-process.
    # ====================================================================
    if scenario_dict.get("is_baseline", False):
        osw = {
            "weather_file": epw_path,
            "file_paths": file_paths,
            "measure_paths": measure_paths,
            "steps": [prototype_step],
            "name": scenario_name,
        }
        success = run_osw(osw, "run.osw", scenario_run_dir, openstudio_path, scenario_name)
        if not success:
            return None
        # Apply Python reporting measure after simulation
        model_path = os.path.join(scenario_run_dir, "run", "in.osm")
        sql_path = os.path.join(scenario_run_dir, "run", "eplusout.sql")
        if os.path.exists(sql_path):
            print(f"  Applying reporting measure...")
            apply_reporting_measure(model_path, sql_path, measure_dir_path, scenario_name)
    # ====================================================================
    # NON-BASELINE:
    #   Phase 1  create prototype in a _proto/ subfolder
    #   Phase 2  apply Python model measures in-process
    #   Phase 3  OSW with seed_file (no measure steps) â†’ E+ simulation
    #   Phase 4  apply Python reporting measure in-process
    # ====================================================================
    else:
        # --- Phase 1: create prototype in isolated subfolder ---
        proto_dir = os.path.join(scenario_run_dir, "_proto")
        proto_osw = {
            "weather_file": epw_path,
            "file_paths": file_paths,
            "measure_paths": measure_paths,
            "steps": [prototype_step],
            "name": f"{scenario_name}_proto",
        }
        success = run_osw(proto_osw, "proto.osw", proto_dir, openstudio_path, f"{scenario_name} [prototype]")
        if not success:
            return None
        proto_model_path = os.path.join(proto_dir, "run", "in.osm")
        if not os.path.exists(proto_model_path):
            print(f"❌ {scenario_name}: prototype model not found at {proto_model_path}")
            return None
        final_model_path = os.path.join(scenario_run_dir, "model_to_run.osm")
        shutil.copy2(proto_model_path, final_model_path)
        # --- Phase 2: apply Python model measures ---
        translator = openstudio.osversion.VersionTranslator()
        loaded_model = translator.loadModel(openstudio.toPath(final_model_path))
        if not loaded_model.is_initialized():
            print(f"❌ {scenario_name}: failed to load prototype model")
            return None
        model = loaded_model.get()
        wall_args = None
        roof_args = None
        window_args = None
        door_args = None
        window_upgrade_status = None
        if scenario_dict.get("wall_r_value"):
            wall_args = {
                "r_value": float(scenario_dict["wall_r_value"]),
                "analysis_period": 30,
                "gwp_statistic": "median",
                "api_key": EC3_API_TOKEN or "",
                "insulation_material_type": "Blown Fiberglass",
                "insulation_material_lifetime": 30,
                "insulation_thermal_conductivity": 0.0,
                "insulation_material_density": 0.0,
            }
            print(f"  Applying wall insulation (R={scenario_dict['wall_r_value']})...")
            if not apply_python_measure(model, Path(measure_dir_path) / "IncreaseInsulationRValueForExteriorWalls", "IncreaseInsulationRValueForExteriorWalls", wall_args):
                print(f"  ❌ {scenario_name}: wall measure failed")
                del model
                return None
        if scenario_dict.get("roof_r_value"):
            roof_args = {
                "r_value": float(scenario_dict["roof_r_value"]),
                "analysis_period": 30,
                "gwp_statistic": "median",
                "api_key": EC3_API_TOKEN or "",
                "insulation_material_type": "Blown Fiberglass",
                "insulation_material_lifetime": 30,
                "insulation_thermal_conductivity": 0.0,
                "insulation_material_density": 0.0,
            }
            print(f"  Applying roof insulation (R={scenario_dict['roof_r_value']})...")
            if not apply_python_measure(model, Path(measure_dir_path) / "IncreaseInsulationRValueForRoofs", "IncreaseInsulationRValueForRoofs", roof_args):
                print(f"  ❌ {scenario_name}: roof measure failed")
                del model
                return None
        if scenario_dict.get("window_u_factor"):
            if EC3_API_TOKEN is None:
                print(f"⚠️  EC3 API token not found, skipping window enhancement")
            else:
                u_factor = float(scenario_dict["window_u_factor"])
                num_panes = 2 if u_factor >= 0.30 else 3
                window_args = {
                    "glass_option": "provide user_num_panes",
                    "user_num_panes": num_panes,
                    "space_infiltration_reduction_percent": 50.0,
                    "glass_pane_thickness": 0.003,
                    "gap_thickness": 0.013,
                    "glass_solar_transmittance": 0.0,
                    "glass_visible_transmittance": 0.0,
                    "glass_front_emissivity": 0.0,
                    "glass_back_emissivity": 0.0,
                    "glass_front_solar_reflectance": 0.0,
                    "glass_back_solar_reflectance": 0.0,
                    "glass_front_visible_reflectance": 0.0,
                    "glass_back_visible_reflectance": 0.0,
                    "analysis_period": 30,
                    "glass_lifetime": 15,
                    "wf_lifetime": 15,
                    "caulking_lifetime": 10,
                    "film_lifetime": 10,
                    "weatherstrip_lifetime": 10,
                    "wf_option": "none",
                    "caulking_option": "acrylic",
                    "caulking_thickness": 0.008,
                    "film_option": "none",
                    "film_visible_transmittance": 0.0,
                    "film_solar_transmittance": 0.0,
                    "film_thermal_emissivity": 0.0,
                    "film_thermal_resistance": 0.0,
                    "weatherstrip_option": "none",
                    "length_per_unit": 5.1816,
                    "secondary_glazing_option": "none",
                    "api_key": EC3_API_TOKEN,
                    "gwp_statistic": "median",
                }
                print(f"  Applying window enhancement (U={scenario_dict['window_u_factor']}, {num_panes} panes)...")
                if not apply_python_measure(model, Path(measure_dir_path) / "window_enhancement", "WindowEnhancement", window_args):
                    print(f"  ❌ {scenario_name}: window measure failed")
                    del model
                    return None
                window_upgrade_status = detect_window_upgrade_status(model, num_panes)
        if scenario_dict.get("door_option"):
            if EC3_API_TOKEN is None:
                print(f"⚠️ EC3 API token not found, skipping door enhancement")
            else:
                door_args = {
                    "space_infiltration_reduction_percent": 30.0,
                    "alter_coef": False,
                    "door_area_per_unit": 1.95,
                    "analysis_period": 30,
                    "door_bottom_seal_option": "automatic door bottom",
                    "door_top_side_seal_option": "jamb weatherstrip",
                    "door_option": scenario_dict["door_option"],
                    "strip_lifetime": 15,
                    "door_lifetime": 30,
                    "gwp_statistic": "median",
                    "api_key": EC3_API_TOKEN,
                    "length_per_unit_bottom_side": 0.9144,
                    "length_per_unit_other_sides": 5.1816,
                    "door_thermal_conductivity": 0.0,
                    "door_density": 0.0,
                    "door_thickness": 0.0,
                }
                print(f"  Applying door enhancement (door={scenario_dict['door_option']})...")
                if not apply_python_measure(model, Path(measure_dir_path) / "door_enhancement", "DoorEnhancement", door_args):
                    print(f"  ❌ {scenario_name}: door measure failed")
                    del model
                    return None
        renovation_details = summarize_renovation_details(
            scenario_name,
            wall_args=wall_args,
            roof_args=roof_args,
            window_args=window_args,
            door_args=door_args,
            window_upgrade_status=window_upgrade_status,
        )
        model.getSite().additionalProperties().setFeature("renovation_details", renovation_details)
        model.save(openstudio.toPath(final_model_path), True)
        del model
        # --- Phase 3: OSW with seed to run EnergyPlus (no measure steps) ---
        sim_osw = {
            "weather_file": epw_path,
            "seed_file": final_model_path,
            "file_paths": file_paths,
            "measure_paths": measure_paths,
            "steps": [],
            "name": scenario_name,
        }
        success = run_osw(sim_osw, "run.osw", scenario_run_dir, openstudio_path, scenario_name)
        if not success:
            return None
        # --- Phase 4: Apply Python reporting measure after simulation ---
        model_path = os.path.join(scenario_run_dir, "run", "in.osm")
        sql_path = os.path.join(scenario_run_dir, "run", "eplusout.sql")
        if os.path.exists(sql_path):
            print(f"  Applying reporting measure...")
            apply_reporting_measure(model_path, sql_path, measure_dir_path, scenario_name)
    print(f"✅ Completed: {scenario_name}")
    return scenario_name
# =========================
# SCENARIO GENERATION
# =========================
def generate_scenarios(
    cities,
    building_types,
    custom_combos=None,
):
    """
    Generate scenarios:
    - Always includes baseline
    - Custom explicit combinations via custom_combos
    """
    scenarios = []
    # 1) BASELINE
    for city, building_type in product(cities, building_types):
        scenarios.append({
            "is_baseline": True,
            "city": city,
            "building_type": building_type,
            "wall_r_value": None,
            "roof_r_value": None,
            "window_u_factor": None,
            "door_option": None,
        })
    # 2) CUSTOM EXPLICIT COMBINATIONS
    if custom_combos:
        for city, building_type, combo in product(cities, building_types, custom_combos):
            scenarios.append({
                "is_baseline": False,
                "city": city,
                "building_type": building_type,
                "wall_r_value": combo.get("wall_r_value"),
                "roof_r_value": combo.get("roof_r_value"),
                "window_u_factor": combo.get("window_u_factor"),
                "door_option": combo.get("door_option"),
            })
    return scenarios
# =========================
# POSTPROCESS: COLLECT RESULTS
# =========================
def _read_props_to_dict(props, prefix=""):
    """Helper: read all features from an AdditionalProperties object into a dict."""
    result = {}
    for name in props.featureNames():
        val = props.getFeatureAsString(name)
        if val.is_initialized():
            v = val.get()
            try:
                result[prefix + name] = float(v)
            except ValueError:
                result[prefix + name] = v
    return result
def extract_additional_properties_from_osm(osm_path):
    """
    Extract AdditionalProperties from an OSM file.
    Reads from: Building, Site, Facility, SimulationControl, SizingParameters.
    Returns a flat dict of all found properties.
    """
    try:
        translator = openstudio.osversion.VersionTranslator()
        translator.setAllowNewerVersions(True)
        loaded_model = translator.loadModel(openstudio.toPath(str(osm_path)))
        if not loaded_model.is_initialized():
            return {}
        model = loaded_model.get()
        prop_dict = {}
        # Building — measure inputs (measure_name, analysis_period_years, gwp_statistic, etc.)
        prop_dict.update(_read_props_to_dict(model.getBuilding().additionalProperties()))
        prop_dict["building_area_m2"] = model.getBuilding().floorArea()
        # Site — reno details + operating cost/emissions from reporting measure
        prop_dict.update(_read_props_to_dict(model.getSite().additionalProperties()))
        # Facility — GWP factors
        prop_dict.update(_read_props_to_dict(model.getFacility().additionalProperties()))
        # SimulationControl — embodied carbon results
        prop_dict.update(_read_props_to_dict(model.getSimulationControl().additionalProperties()))
        # SizingParameters — material properties (lifetimes, densities, etc.)
        prop_dict.update(_read_props_to_dict(model.getSizingParameters().additionalProperties()))
        del model
        return prop_dict
    except Exception as e:
        print(f"  ❌  Failed to extract properties from {osm_path}: {e}")
        return {}
def collect_results_to_csv(base_run_dir, csv_base_name="parametric_results", city_climate_zones=None):
    """
    Walk all run directories, extract AdditionalProperties from OSM files,
    and collect all results into a comprehensive CSV.
    """
    import pandas as pd
    rows = []
    for scenario_folder in sorted(os.listdir(base_run_dir)):
        scenario_path = os.path.join(base_run_dir, scenario_folder)
        if not os.path.isdir(scenario_path):
            continue
        sql_path = os.path.join(scenario_path, "run", "eplusout.sql")
        if not os.path.exists(sql_path):
            continue
        scenario_parts = scenario_folder.split("_")
        is_baseline = scenario_parts[0] == "baseline"
        city = scenario_parts[-1]
        building_type = scenario_parts[-2]
        climate_zone = city_climate_zones.get(city, "") if city_climate_zones else ""
        renovation_details = summarize_renovation_details(scenario_folder)
        row = {
            "scenario_name": scenario_folder,
            "is_baseline": is_baseline,
            "city": city,
            "building_type": building_type,
            "climate_zone": climate_zone,
            "renovation_details": renovation_details,
        }
        osm_path = os.path.join(scenario_path, "run", "in.osm")
        if os.path.exists(osm_path):
            print(f"  Extracting properties from {scenario_folder}...")
            row.update(extract_additional_properties_from_osm(osm_path))
        rows.append(row)
    if not rows:
        print("\n📊 No simulation results found.")
        return None
    df_results = pd.DataFrame(rows)
    priority_cols = ["scenario_name", "is_baseline", "city", "building_type", "climate_zone", "building_area_m2", "renovation_details"]
    existing_priority = [c for c in priority_cols if c in df_results.columns]
    remaining = sorted([c for c in df_results.columns if c not in existing_priority])
    df_results = df_results[existing_priority + remaining]
    csv_path = os.path.join(base_run_dir, f"{csv_base_name}.csv")
    df_results.to_csv(csv_path, index=False)
    print(f"\n📊 Results CSV: {csv_path}")
    print(f"   Total scenarios: {len(df_results)}")
    print(f"   Columns: {len(df_results.columns)}")
    return df_results
def extract_total_site_energy_gj(sql_path):
    """Extract Total Site Energy [GJ] from EnergyPlus SQL tabular data."""
    try:
        with sqlite3.connect(str(sql_path)) as conn:
            cur = conn.cursor()
            cur.execute(
                """
                SELECT Value
                FROM TabularDataWithStrings
                WHERE lower(ReportName) = 'annualbuildingutilityperformancesummary'
                  AND lower(TableName) = 'site and source energy'
                  AND lower(RowName) = 'total site energy'
                LIMIT 1
                """
            )
            row = cur.fetchone()
            if row and row[0] not in (None, ""):
                return float(row[0])
    except Exception as e:
        print(f"    ❌ Failed to read total site energy from SQL {sql_path}: {e}")
    return None
def get_prop_value(props, name):
    """Helper to safely extract value from AdditionalProperties by type."""
    if props.getFeatureAsDouble(name).is_initialized():
        return props.getFeatureAsDouble(name).get()
    if props.getFeatureAsString(name).is_initialized():
        return props.getFeatureAsString(name).get()
    if props.getFeatureAsInteger(name).is_initialized():
        return props.getFeatureAsInteger(name).get()
    return None
def extract_scenario_data(osm_path, scenario_name):
    """
    Loads an OSM and extracts target properties from AdditionalProperties.
    """
    results = {"scenario": scenario_name}
    vt = openstudio.osversion.VersionTranslator()
    model_ptr = vt.loadModel(openstudio.toPath(str(osm_path)))
    if not model_ptr.is_initialized():
        print(f"  ❌ Failed to load: {osm_path.name}")
        return None
    model = model_ptr.get()
    results["building_area_m2"] = model.getBuilding().floorArea()
    inferred_wall_args = infer_wall_args_from_scenario_name(scenario_name)
    inferred_roof_args = infer_roof_args_from_scenario_name(scenario_name)
    inferred_window_args = infer_window_args_from_scenario_name(scenario_name)
    inferred_door_args = infer_door_args_from_scenario_name(scenario_name)
    window_upgrade_status = None
    if inferred_window_args:
        window_upgrade_status = detect_window_upgrade_status(model, inferred_window_args["user_num_panes"])

    results["renovation_details"] = summarize_renovation_details(
        scenario_name,
        wall_args=inferred_wall_args,
        roof_args=inferred_roof_args,
        window_args=inferred_window_args,
        door_args=inferred_door_args,
        window_upgrade_status=window_upgrade_status,
    )
    found_any = True
    # 1. Embodied Carbon — SimulationControl.additionalProperties()
    sim_props = model.getSimulationControl().additionalProperties()
    ec_keys = [
        "wall_insulation_total_additional_embodied_carbon_kg",
        "roof_insulation_total_additional_embodied_carbon_kg",
        "window_enhancement_total_additional_embodied_carbon_kg",
        "door_enhancement_total_additional_embodied_carbon_kg",
    ]
    for key in ec_keys:
        if key in sim_props.featureNames():
            val = get_prop_value(sim_props, key)
            if val is not None:
                results[key] = val
                found_any = True
    # 2. Operating Cost and Emissions — Site.additionalProperties()
    site_props = model.getSite().additionalProperties()
    op_keys = [
        "annual_electricity_cost_usd",
        "annual_gas_cost_usd",
        "annual_electricity_operating_emissions_kg_co2e",
        "annual_gas_operating_emissions_kg_co2e",
        "total_site_energy_gj",
    ]
    for key in op_keys:
        if key in site_props.featureNames():
            val = get_prop_value(site_props, key)
            if val is not None:
                results[key] = val
                found_any = True
    # Total site energy (GJ): prefer Site AdditionalProperties; fallback to SQL tabular data
    site_energy_keys = ["total_site_energy_GJ", "total_site_energy_gj"]
    for key in site_energy_keys:
        if key in site_props.featureNames():
            val = get_prop_value(site_props, key)
            if val is not None:
                results["total_site_energy_gj"] = val
                found_any = True
                break
    if "total_site_energy_gj" not in results:
        sql_path = osm_path.parent / "eplusout.sql"
        if sql_path.exists():
            total_site_energy = extract_total_site_energy_gj(sql_path)
            if total_site_energy is not None:
                results["total_site_energy_gj"] = total_site_energy
                found_any = True
    # Explicit reno detail keys written by envelope measures
    reno_keys = [
        "total_renovated_window_area_m2",
        "total_renovated_glazing_area_m2",
        "total_renovated_frame_area_m2",
        "total_renovated_perimeter_m",
        "total_renovated_caulking_volume_m3",
        "total_renovated_weatherstrip_length_m",
        "total_renovated_door_area_m2",
        "total_renovated_sealing_bottom_length_m",
        "total_renovated_sealing_side_length_m",
    ]
    for key in reno_keys:
        if key in site_props.featureNames():
            val = get_prop_value(site_props, key)
            if val is not None:
                results[key] = val
                found_any = True
    return results if found_any else None
def generate_parametric_recap(target_path, city_climate_zones=None):
    """
    Main function to run the extraction and generate parametric_results.csv
    """
    root_path = Path(target_path)
    if not root_path.exists():
        print(f"Error: Path '{target_path}' does not exist.")
        return
    all_data = []
    all_headers = set()
    print("=" * 80)
    print(f"GENERATING PARAMETRIC RECAP FROM: {root_path}")
    print("=" * 80)
    scenario_dirs = [p for p in sorted(root_path.iterdir()) if p.is_dir()]
    for scenario_dir in scenario_dirs:
        scenario = scenario_dir.name
        primary_osm = scenario_dir / "run" / "in.osm"
        secondary_osm = scenario_dir / "run" / "in_modified.osm"
        if primary_osm.exists():
            osm_path = primary_osm
        elif secondary_osm.exists():
            osm_path = secondary_osm
        else:
            continue
        data = extract_scenario_data(osm_path, scenario)
        if data:
            s_parts = scenario.split("_")
            data["city"] = s_parts[-1]
            data["building_type"] = s_parts[-2]
            data["climate_zone"] = city_climate_zones.get(s_parts[-1], "") if city_climate_zones else ""
            all_data.append(data)
            all_headers.update(data.keys())
    if not all_data:
        print("\n No matching data found.")
        return
    fixed_headers = [
        "scenario",
        "city",
        "building_type",
        "climate_zone",
        "building_area_m2",
        "renovation_details",
        "annual_electricity_cost_usd",
        "annual_gas_cost_usd",
        "annual_electricity_operating_emissions_kg_co2e",
        "annual_gas_operating_emissions_kg_co2e",
        "total_site_energy_gj",
        "total_renovated_window_area_m2",
        "total_renovated_glazing_area_m2",
        "total_renovated_frame_area_m2",
        "total_renovated_perimeter_m",
        "total_renovated_caulking_volume_m3",
        "total_renovated_weatherstrip_length_m",
        "total_renovated_door_area_m2",
        "total_renovated_sealing_bottom_length_m",
        "total_renovated_sealing_side_length_m",
        "wall_insulation_total_additional_embodied_carbon_kg",
        "roof_insulation_total_additional_embodied_carbon_kg",
        "window_enhancement_total_additional_embodied_carbon_kg",
        "door_enhancement_total_additional_embodied_carbon_kg",
    ]
    extra_headers = sorted([h for h in all_headers if h not in fixed_headers and h != "scenario"])
    fieldnames = [h for h in fixed_headers if h in all_headers | {"scenario"}] + extra_headers
    desired_scenarios = [
        generate_scenario_name(s)
        for s in generate_scenarios(cities=CITIES, building_types=BUILDING_TYPES, custom_combos=CUSTOM_COMBOS)
    ]
    scenario_order = {name: idx for idx, name in enumerate(desired_scenarios)}
    all_data = sorted(
        all_data,
        key=lambda row: (scenario_order.get(str(row.get("scenario", "")), len(scenario_order)), str(row.get("scenario", ""))),
    )
    csv_path = root_path / "parametric_results.csv"
    with open(csv_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(all_data)
    print("\n" + "=" * 80)
    print(f"COMPLETE: {len(all_data)} scenarios successfully processed.")
    print(f"Report saved to: {csv_path}")
    print("=" * 80)
# =========================
# GLOBAL SETTINGS
# =========================
RUN_NAME = "run_test_001"
def detect_openstudio_cli_path():
    env_path = os.environ.get("OPENSTUDIO_PATH")
    candidates = [
        env_path,
        "C:/Program Files/openstudio-3.11.0/bin/openstudio.exe",
        "/Applications/OpenStudio-3.11.0/bin/openstudio",
        "openstudio",
    ]
    for candidate in candidates:
        if not candidate:
            continue
        if candidate == "openstudio" or os.path.exists(candidate):
            return candidate
    return "openstudio"
OPENSTUDIO_PATH = detect_openstudio_cli_path()
OVERWRITE_EXISTING = False
notebook_dir = Path.cwd()
base_weather_path = str(notebook_dir / "weather")
measure_dir_path = str(notebook_dir.parent / "measures")
base_run_dir = str(notebook_dir / "simulations" / RUN_NAME)
city_climate_zones = {
    "Amarillo":     "ASHRAE 169-2013-3B",
    # "Atlanta":      "ASHRAE 169-2013-3A",
    # "Baltimore":    "ASHRAE 169-2013-4A",
    # "Chicago":      "ASHRAE 169-2013-5A",
    # "Denver":       "ASHRAE 169-2013-5B",
    # "Duluth":       "ASHRAE 169-2013-7A",
    # "ElPaso":       "ASHRAE 169-2013-3B",
    # "Fairbanks":    "ASHRAE 169-2013-8A",
    # "Helena":       "ASHRAE 169-2013-6B",
    # "Houston":      "ASHRAE 169-2013-2A",
    # "Miami":        "ASHRAE 169-2013-1A",
    # "Minneapolis":  "ASHRAE 169-2013-6A",
    # "Phoenix":      "ASHRAE 169-2013-2B",
    # "PortAngeles":  "ASHRAE 169-2013-4C",
    # "Portland":     "ASHRAE 169-2013-4C",
    # "SanFrancisco": "ASHRAE 169-2013-3C",
}
# =========================
# PARAMETRIC STUDY CONFIGURATION
# =========================
CITIES = list(city_climate_zones.keys())
BUILDING_TYPES = [
    "SmallOffice",
    # "MediumOffice",
    # "LargeOffice",
    # "SmallHotel",
    # "LargeHotel",
    # "Warehouse",
    # "RetailStandalone",
    # "RetailStripmall",
    # "PrimarySchool",
    # "SecondarySchool",
]
TEMPLATE = "90.1-2010"
# =========================
# CUSTOM COMBINATION SCENARIOS
# Scenario 1: Baseline  – auto-generated (no entry needed here)
# Scenario 2: Wall + Door
# Scenario 3: Roof + Window
# Scenario 4: Wall + Door + Roof + Window (all 4 measures)
# =========================
CUSTOM_COMBOS = [
    # Scenario 1: Window only
    {
        "wall_r_value":    None,
        "roof_r_value":    None,
        "window_u_factor": 0.20,
        "door_option":     None,
    },
    # Scenario 2: Door only
    {
        "wall_r_value":    None,
        "roof_r_value":    None,
        "window_u_factor": None,
        "door_option":     "wooden door",
    },
    # Scenario 3: Wall only
    {
        "wall_r_value":    30,
        "roof_r_value":    None,
        "window_u_factor": None,
        "door_option":     None,
    },
    # Scenario 4: Roof only
    {
        "wall_r_value":    None,
        "roof_r_value":    40,
        "window_u_factor": None,
        "door_option":     None,
    },
    # Scenario 5: Wall + Door only
    {
        "wall_r_value":    30,
        "roof_r_value":    None,
        "window_u_factor": None,
        "door_option":     "wooden door",
    },
    # Scenario 6: Roof + Window only
    {
        "wall_r_value":    None,
        "roof_r_value":    40,
        "window_u_factor": 0.20,
        "door_option":     None,
    },
    # Scenario 7: All 4 measures – Wall + Door + Roof + Window
    {
        "wall_r_value":    30,
        "roof_r_value":    40,
        "window_u_factor": 0.20,
        "door_option":     "wooden door",
    },
]
def scenario_output_exists(base_run_dir, scenario_dict):
    scenario_name = generate_scenario_name(scenario_dict)
    sql_path = os.path.join(base_run_dir, scenario_name, "run", "eplusout.sql")
    return os.path.exists(sql_path)
# =========================
# MAIN - RUN PARAMETRIC STUDY
# =========================
if __name__ == "__main__":
    print("\n" + "=" * 70)
    print("PARAMETRIC STUDY: BUILDING ENERGY EFFICIENCY MEASURES")
    print(f"Using OpenStudio: {OPENSTUDIO_PATH}")
    print(f"Run Name: {RUN_NAME}")
    print(f"Output Directory: {base_run_dir}")
    print("=" * 70)
    print("\n🔹 Generating scenarios...")
    scenarios = generate_scenarios(
        cities=CITIES,
        building_types=BUILDING_TYPES,
        custom_combos=CUSTOM_COMBOS,
    )
    total_sims = len(scenarios)
    baseline_count = sum(1 for s in scenarios if s["is_baseline"])
    individual_wall = sum(1 for s in scenarios if not s["is_baseline"] and s["wall_r_value"] and not s["roof_r_value"] and not s["window_u_factor"] and not s["door_option"])
    individual_roof = sum(1 for s in scenarios if not s["is_baseline"] and s["roof_r_value"] and not s["wall_r_value"] and not s["window_u_factor"] and not s["door_option"])
    individual_window = sum(1 for s in scenarios if not s["is_baseline"] and s["window_u_factor"] and not s["wall_r_value"] and not s["roof_r_value"] and not s["door_option"])
    individual_door = sum(1 for s in scenarios if not s["is_baseline"] and s["door_option"] and not s["wall_r_value"] and not s["roof_r_value"] and not s["window_u_factor"])
    all_measures = sum(1 for s in scenarios if not s["is_baseline"] and sum([bool(s["wall_r_value"]), bool(s["roof_r_value"]), bool(s["window_u_factor"]), bool(s["door_option"])]) > 1)
    print(f"\n🔹 Total scenarios: {total_sims}")
    print(f"   - Cities: {len(CITIES)}")
    print(f"   - Building Types: {len(BUILDING_TYPES)}")
    print(f"\n   Breakdown:")
    print(f"   - Baseline: {baseline_count}")
    print(f"   - Wall only: {individual_wall}")
    print(f"   - Roof only: {individual_roof}")
    print(f"   - Window only: {individual_window}")
    print(f"   - Door only: {individual_door}")
    print(f"   - Combined (≥2 measures): {all_measures}")
    print("=" * 70)
    sim_count = 0
    start_time = time.time()
    successful_scenarios = []
    failed_scenarios = []
    all_selected_have_results = all(scenario_output_exists(base_run_dir, s) for s in scenarios)
    skip_simulation_run = all_selected_have_results and CUSTOM_COMBOS and not OVERWRITE_EXISTING
    if skip_simulation_run:

        print("\n⚠️  Existing simulation outputs detected for selected scenarios.")
        print("   Skipping simulation run and regenerating CSV only.")
        successful_scenarios = [generate_scenario_name(s) for s in scenarios]
    else:
        for scenario in scenarios:
            sim_count += 1
            city = scenario["city"]
            building_type = scenario["building_type"]
            climate_zone = city_climate_zones.get(city, "ASHRAE 169-2013-5A")
            scenario_name = generate_scenario_name(scenario)
            print(f"\n[{sim_count}/{total_sims}] {scenario_name}")
            sim_start = time.time()
            result = create_simulation(
                city=city,
                base_run_dir=base_run_dir,
                measure_dir_path=measure_dir_path,
                base_weather_path=base_weather_path,
                scenario_dict=scenario,
                overwrite_existing=OVERWRITE_EXISTING,
                building_type=building_type,
                template=TEMPLATE,
                climate_zone=climate_zone,
                openstudio_path=OPENSTUDIO_PATH,
            )
            if result:
                successful_scenarios.append(result)
            else:
                failed_scenarios.append(scenario_name)

            sim_elapsed = time.time() - sim_start
            print(f"   ⏱️  Time: {sim_elapsed/60:.1f} min")
    total_elapsed = time.time() - start_time
    print("\n" + "=" * 70)
    print("SIMULATION SUMMARY")
    print("=" * 70)

    print(f"   ⏱️  Total time: {total_elapsed/60:.1f} min ({total_elapsed/3600:.2f} hours)")
    print(f"   ✅ Successful: {len(successful_scenarios)}/{total_sims}")
    print(f"   ❌ Failed: {len(failed_scenarios)}/{total_sims}")

    if failed_scenarios:
        print("\nFailed scenarios:")
        for failed in failed_scenarios:
            print(f"  - {failed}")
    # Collect results
    print("\n" + "=" * 70)
    print("COLLECTING RESULTS FROM OSM FILES")
    print("=" * 70)
    generate_parametric_recap(f"./simulations/{RUN_NAME}", city_climate_zones)
    print("\n✅ Parametric study complete!")



PARAMETRIC STUDY: BUILDING ENERGY EFFICIENCY MEASURES
Using OpenStudio: C:/Program Files/openstudio-3.11.0/bin/openstudio.exe
Run Name: run_test_001
Output Directory: c:\All repos\openstudio-ee-gem\lib\parametric_run\simulations\run_test_001

🔹 Generating scenarios...

🔹 Total scenarios: 8
   - Cities: 1
   - Building Types: 1

   Breakdown:
   - Baseline: 1
   - Wall only: 1
   - Roof only: 1
   - Window only: 1
   - Door only: 1
   - Combined (≥2 measures): 3

[1/8] baseline_SmallOffice_Amarillo
⚠️  Skipping baseline_SmallOffice_Amarillo - simulation already exists
   ⏱️  Time: 0.0 min

[2/8] window_u0.2_SmallOffice_Amarillo
⚠️  Skipping window_u0.2_SmallOffice_Amarillo - simulation already exists
   ⏱️  Time: 0.0 min

[3/8] door_wooden_d_SmallOffice_Amarillo
⚠️  Skipping door_wooden_d_SmallOffice_Amarillo - simulation already exists
   ⏱️  Time: 0.0 min

[4/8] wall_r30_SmallOffice_Amarillo
  Applying wall insulation (R=30)...
[EC3_lookup] API token loaded from c:\All repos\openstud

# Spider Chart visualizing embodied/operational carbon/cost

In [11]:
# Spider chart for parametric_results.csv (4 scenarios)
from pathlib import Path
import textwrap
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display

base_dir = Path.cwd() / "simulations" / RUN_NAME
csv_candidates = [
    base_dir / "parametric_results.csv",
]

csv_path = next((p for p in csv_candidates if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError(f"Could not find CSV in: {csv_candidates}")

df = pd.read_csv(csv_path)
baseline_first_mask = df["scenario"].astype(str).str.contains("baseline", case=False, na=False)
df = pd.concat([df[baseline_first_mask], df[~baseline_first_mask]], ignore_index=True)
scenario_values = df["scenario"].astype(str).tolist()
scenario_display_map = {}
scenario_counter = 1
for name in scenario_values:
    if "baseline" in str(name).lower():
        scenario_display_map[name] = "Baseline"
    elif name not in scenario_display_map:
        scenario_display_map[name] = f"Scenario {scenario_counter}"
        scenario_counter += 1

required_cols = [
    "scenario",
    "window_enhancement_total_additional_embodied_carbon_kg",
    "door_enhancement_total_additional_embodied_carbon_kg",
    "wall_insulation_total_additional_embodied_carbon_kg",
    "roof_insulation_total_additional_embodied_carbon_kg",
    "annual_electricity_operating_emissions_kg_co2e",
    "annual_gas_operating_emissions_kg_co2e",
    "annual_electricity_cost_usd",
    "annual_gas_cost_usd",
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

numeric_cols = [c for c in required_cols if c != "scenario"]
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)

df["total_additional_embodied_carbon_kg"] = (
    df["window_enhancement_total_additional_embodied_carbon_kg"]
    + df["door_enhancement_total_additional_embodied_carbon_kg"]
    + df["wall_insulation_total_additional_embodied_carbon_kg"]
    + df["roof_insulation_total_additional_embodied_carbon_kg"]
)

metrics = [
    "total_additional_embodied_carbon_kg",
    "annual_electricity_operating_emissions_kg_co2e",
    "annual_gas_operating_emissions_kg_co2e",
    "annual_electricity_cost_usd",
    "annual_gas_cost_usd",
]

metric_labels = {
    "total_additional_embodied_carbon_kg": "Total Embodied Carbon (kgCO2e)",
    "annual_electricity_operating_emissions_kg_co2e": "Annual Electricity Emissions (kgCO2e)",
    "annual_gas_operating_emissions_kg_co2e": "Annual Gas Emissions (kgCO2e)",
    "annual_electricity_cost_usd": "Annual Electricity Cost (USD)",
    "annual_gas_cost_usd": "Annual Gas Cost (USD)",
}

def wrap_label(text, width=22):
    parts = textwrap.wrap(str(text), width=width)
    return "<br>".join(parts) if parts else str(text)

theta_raw = [metric_labels[m] for m in metrics]
theta = [wrap_label(label, width=22) for label in theta_raw]
max_vals = df[metrics].max().replace(0, 1.0)

fig = go.Figure()
for _, row in df.iterrows():
    raw_vals = [float(row[m]) for m in metrics]
    norm_vals = [(float(row[m]) / float(max_vals[m])) for m in metrics]

    fig.add_trace(
        go.Scatterpolar(
            theta=theta + [theta[0]],
            r=norm_vals + [norm_vals[0]],
            customdata=raw_vals + [raw_vals[0]],
            name=scenario_display_map.get(str(row["scenario"]), str(row["scenario"])),
            hovertemplate="<b>%{theta}</b><br>Normalized: %{r:.3f}<br>Value: %{customdata:,.2f}<extra>%{fullData.name}</extra>",
        )
    )

fig.update_layout(
    title="Parametric Scenario Comparison (Normalized Spider Chart)",
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    showlegend=True,
)

html_output = base_dir / "parametric_spider_chart.html"
fig.write_html(str(html_output))
print(f"Spider chart saved to: {html_output}")
print("Interactive chart and summary table are displayed below and chart is also saved as HTML.")

# Inline render right after running the cell (avoids plotly mime renderer dependency)
display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))

# Results table matching spider chart metrics
table_df = df[["scenario"] + metrics].rename(columns=metric_labels)
table_df["scenario"] = table_df["scenario"].astype(str).map(lambda s: scenario_display_map.get(s, s))
table_df = table_df.round(2)
display(HTML("<h4>Spider Chart Data Table</h4>"))
display(table_df)

Spider chart saved to: c:\All repos\openstudio-ee-gem\lib\parametric_run\simulations\run_test_001\parametric_spider_chart.html
Interactive chart and summary table are displayed below and chart is also saved as HTML.


,scenario,Total Embodied Carbon (kgCO2e),Annual Electricity Emissions (kgCO2e),Annual Gas Emissions (kgCO2e),Annual Electricity Cost (USD),Annual Gas Cost (USD)
0,Baseline,0.00,19257.01,4.61,9384.46,0.88
1,Scenario 1,27.69,18765.13,34.36,9514.63,6.52
2,Scenario 2,423.05,18773.96,39.23,9546.07,7.44
3,Scenario 3,2307.08,18619.35,39.15,9436.31,7.43
4,Scenario 4,11231.16,18704.60,47.76,9495.10,9.06
5,Scenario 5,2730.14,18505.28,31.41,9339.86,5.96
6,Scenario 6,11231.16,18565.62,34.00,9369.53,6.45
7,Scenario 7,11654.21,18230.21,20.71,9046.80,3.93


# Comprehensive HTML report

In [15]:
from pathlib import Path
from datetime import datetime
import pandas as pd

def _to_num(df, col):
    if col in df.columns:
        return pd.to_numeric(df[col], errors="coerce").fillna(0.0)
    return pd.Series([0.0] * len(df), index=df.index)

def generate_html_report(df, html_report_path, run_name="run"):
    scenario_col = "scenario" if "scenario" in df.columns else "scenario_name"
    if scenario_col not in df.columns:
        raise ValueError("CSV must include 'scenario' or 'scenario_name' column.")

    baseline_first_mask = df[scenario_col].astype(str).str.contains("baseline", case=False, na=False)
    df = pd.concat([df[baseline_first_mask], df[~baseline_first_mask]], ignore_index=True)

    scenario_values = df[scenario_col].astype(str).tolist()
    scenario_display_map = {}
    scenario_counter = 1
    for name in scenario_values:
        if "baseline" in str(name).lower():
            scenario_display_map[name] = "Baseline"
        elif name not in scenario_display_map:
            scenario_display_map[name] = f"Scenario {scenario_counter}"
            scenario_counter += 1

    elec_cost = _to_num(df, "annual_electricity_cost_usd")
    gas_cost = _to_num(df, "annual_gas_cost_usd")
    elec_emis = _to_num(df, "annual_electricity_operating_emissions_kg_co2e")
    gas_emis = _to_num(df, "annual_gas_operating_emissions_kg_co2e")
    site_energy = _to_num(df, "total_site_energy_gj")

    analysis_period_col_candidates = ["analysis_period", "wall_analysis_period", "window_analysis_period", "roof_analysis_period", "door_analysis_period", "analysis_period_years", "analysis_period_yrs"]
    available_analysis_period_col = next((c for c in analysis_period_col_candidates if c in df.columns), None)
    default_embodied_analysis_period_years = 30.0
    if available_analysis_period_col:
        raw_analysis_period = pd.to_numeric(df[available_analysis_period_col], errors="coerce")
        valid_periods = raw_analysis_period[raw_analysis_period > 0]
        if not valid_periods.empty:
            default_embodied_analysis_period_years = float(valid_periods.iloc[0])
        analysis_period_years = raw_analysis_period.where(raw_analysis_period > 0, default_embodied_analysis_period_years).fillna(default_embodied_analysis_period_years)
    else:
        analysis_period_years = pd.Series([default_embodied_analysis_period_years] * len(df), index=df.index)

    wall_ec = _to_num(df, "wall_insulation_total_additional_embodied_carbon_kg")
    roof_ec = _to_num(df, "roof_insulation_total_additional_embodied_carbon_kg")
    window_ec = _to_num(df, "window_enhancement_total_additional_embodied_carbon_kg")
    door_ec = _to_num(df, "door_enhancement_total_additional_embodied_carbon_kg")

    report = pd.DataFrame({
        "scenario": df[scenario_col].astype(str),
        "annual_cost_usd": elec_cost + gas_cost,
        "annual_emissions_kg": elec_emis + gas_emis,
        "total_site_energy_gj": site_energy,
        "analysis_period_years": analysis_period_years,
        "wall_ec": wall_ec,
        "roof_ec": roof_ec,
        "window_ec": window_ec,
        "door_ec": door_ec,
    })
    report["embodied_carbon_kg"] = report[["wall_ec", "roof_ec", "window_ec", "door_ec"]].sum(axis=1)

    baseline_mask = report["scenario"].str.contains("baseline", case=False, na=False)
    baseline = report[baseline_mask].head(1)
    if baseline.empty:
        baseline = report.head(1)

    b = baseline.iloc[0]
    embodied_analysis_period_years = max(float(default_embodied_analysis_period_years), 1.0)
    operational_analysis_period_years = 1.0

    comparison_df = report[~baseline_mask].copy()
    if comparison_df.empty:
        comparison_df = report.head(0).copy()

    comparison_df["cost_delta"] = b["annual_cost_usd"] - comparison_df["annual_cost_usd"]
    comparison_df["cost_delta_pct"] = comparison_df["cost_delta"] / b["annual_cost_usd"] * 100.0 if b["annual_cost_usd"] > 0 else 0.0
    comparison_df["emissions_delta"] = b["annual_emissions_kg"] - comparison_df["annual_emissions_kg"]
    comparison_df["emissions_delta_pct"] = comparison_df["emissions_delta"] / b["annual_emissions_kg"] * 100.0 if b["annual_emissions_kg"] > 0 else 0.0

    if comparison_df.empty:
        max_savings = b
        max_emissions_reduction = b
        max_savings_scenario = "No renovation scenarios"
        max_emissions_scenario = "No renovation scenarios"
        max_cost_delta = 0.0
        max_cost_delta_pct = 0.0
        max_emis_delta = 0.0
        max_emis_delta_pct = 0.0
    else:
        max_savings = comparison_df.loc[comparison_df["cost_delta"].idxmax()]
        max_emissions_reduction = comparison_df.loc[comparison_df["emissions_delta"].idxmax()]
        max_savings_scenario = scenario_display_map.get(str(max_savings["scenario"]), str(max_savings["scenario"]))
        max_emissions_scenario = scenario_display_map.get(str(max_emissions_reduction["scenario"]), str(max_emissions_reduction["scenario"]))
        max_cost_delta = float(max_savings["cost_delta"])
        max_cost_delta_pct = float(max_savings["cost_delta_pct"])
        max_emis_delta = float(max_emissions_reduction["emissions_delta"])
        max_emis_delta_pct = float(max_emissions_reduction["emissions_delta_pct"])

    construction_cost_columns = [
        "total_construction_cost_usd",
        "total_additional_construction_cost_usd",
        "construction_cost_usd",
        "additional_construction_cost_usd",
    ]
    available_construction_cost_col = next((c for c in construction_cost_columns if c in df.columns), None)

    if comparison_df.empty:
        min_construction_cost_scenario = "No renovation scenarios"
        min_construction_cost_text = "N/A"
        min_embodied_scenario = "No renovation scenarios"
        min_embodied_text = "N/A"
    else:
        if available_construction_cost_col:
            construction_cost_series = pd.to_numeric(df[available_construction_cost_col], errors="coerce").fillna(float("inf"))
            construction_df = pd.DataFrame({
                "scenario": df[scenario_col].astype(str),
                "construction_cost": construction_cost_series,
            })
            construction_df = construction_df[~construction_df["scenario"].str.contains("baseline", case=False, na=False)]
            valid_construction_df = construction_df[construction_df["construction_cost"] < float("inf")]

            if valid_construction_df.empty:
                min_construction_cost_scenario = "No construction cost data"
                min_construction_cost_text = "N/A"
            else:
                min_construction_row = valid_construction_df.loc[valid_construction_df["construction_cost"].idxmin()]
                min_construction_cost_scenario = scenario_display_map.get(str(min_construction_row["scenario"]), str(min_construction_row["scenario"]))
                min_construction_cost_text = f"${float(min_construction_row['construction_cost']):,.2f}"
        else:
            min_construction_cost_scenario = "Construction cost not in CSV"
            min_construction_cost_text = "N/A"

        min_embodied_row = comparison_df.loc[comparison_df["embodied_carbon_kg"].idxmin()]
        min_embodied_scenario = scenario_display_map.get(str(min_embodied_row["scenario"]), str(min_embodied_row["scenario"]))
        min_embodied_text = f"{float(min_embodied_row['embodied_carbon_kg']):,.2f} kgCO2e"

    savings_count = int((comparison_df["cost_delta"] > 0).sum()) if not comparison_df.empty else 0
    total_compared = int(len(comparison_df))
    savings_share_pct = (savings_count / total_compared * 100.0) if total_compared > 0 else 0.0

    max_cost_class = "positive" if max_cost_delta >= 0 else ""
    max_emis_class = "positive" if max_emis_delta >= 0 else ""

    energy_cost_per_gj = (b["annual_cost_usd"] / b["total_site_energy_gj"]) if b["total_site_energy_gj"] > 0 else 0.0

    max_chart_cost = max(b["annual_cost_usd"], max_savings["annual_cost_usd"], 1.0)
    baseline_cost_w = b["annual_cost_usd"] / max_chart_cost * 100.0
    best_cost_w = max_savings["annual_cost_usd"] / max_chart_cost * 100.0

    max_chart_emis = max(b["annual_emissions_kg"], max_emissions_reduction["annual_emissions_kg"], 1.0)
    baseline_emis_w = b["annual_emissions_kg"] / max_chart_emis * 100.0
    best_emis_w = max_emissions_reduction["annual_emissions_kg"] / max_chart_emis * 100.0

    def money(v):
        return f"${v:,.2f}"

    def num(v):
        return f"{v:,.2f}"

    spider_table_rows = "".join(
        f"<tr><td>{scenario_display_map.get(str(report.iloc[i]['scenario']), str(report.iloc[i]['scenario']))}</td><td>{num(float(report.iloc[i]['embodied_carbon_kg']))}</td><td>{num(float(elec_emis.iloc[i]))}</td><td>{num(float(gas_emis.iloc[i]))}</td><td>{money(float(elec_cost.iloc[i]))}</td><td>{money(float(gas_cost.iloc[i]))}</td></tr>"
        for i in range(len(report))
    )

    if not spider_table_rows:
        spider_table_rows = "<tr><td colspan='6'>No spider chart data found in CSV.</td></tr>"

    if "renovation_details" in df.columns:
        renovation_df = df[[scenario_col, "renovation_details"]].copy()
    else:
        renovation_df = df[[scenario_col]].copy()
        renovation_df["renovation_details"] = "Not available in CSV"

    def _renovation_type_from_scenario_name(raw_name):
        scenario_name_lower = str(raw_name).lower()
        if "baseline" in scenario_name_lower:
            return "baseline"

        types = []
        if "wall_r" in scenario_name_lower:
            types.append("wall")
        if "window_u" in scenario_name_lower:
            types.append("window")
        if "roof_r" in scenario_name_lower:
            types.append("roof")
        if "door_" in scenario_name_lower:
            types.append("door")

        return " + ".join(types) if types else "unknown"

    renovation_df[scenario_col] = renovation_df[scenario_col].astype(str)
    renovation_df["renovation_details"] = renovation_df["renovation_details"].fillna("Not specified").astype(str)
    renovation_df["renovation_type"] = renovation_df[scenario_col].apply(_renovation_type_from_scenario_name)

    renovation_rows = "".join(
        f"<tr><td>{scenario_display_map.get(row[scenario_col], row[scenario_col])}</td><td>{row['renovation_type']}</td><td>{row['renovation_details']}</td></tr>"
        for _, row in renovation_df.iterrows()
    )

    energy_analysis_rows = []
    for _, row in comparison_df.iterrows():
        row_delta = b["total_site_energy_gj"] - row["total_site_energy_gj"]
        row_delta_pct = (row_delta / b["total_site_energy_gj"] * 100.0) if b["total_site_energy_gj"] > 0 else 0.0
        positive_class = "positive" if row_delta >= 0 else ""
        energy_analysis_rows.append(
            f"<tr>"
            f"<td>{scenario_display_map.get(row['scenario'], row['scenario'])}</td>"
            f"<td>Total Site Energy (GJ)</td>"
            f"<td>{num(b['total_site_energy_gj'])}</td>"
            f"<td>{num(row['total_site_energy_gj'])}</td>"
            f"<td class=\"{positive_class}\">{num(row_delta)}</td>"
            f"<td class=\"{positive_class}\">{num(row_delta_pct)}%</td>"
            f"</tr>"
        )
    if energy_analysis_rows:
        energy_analysis_table = "".join(energy_analysis_rows)
    else:
        energy_analysis_table = "<tr><td colspan='6'>No applied renovation scenarios found.</td></tr>"

    stacked_rows = []
    for _, row in report.iterrows():
        scenario_name = scenario_display_map.get(str(row["scenario"]), str(row["scenario"]))
        period_years = max(float(row["analysis_period_years"]), 1.0)
        embodied_per_year = max(float(row["embodied_carbon_kg"]) / period_years, 0.0)
        operational_per_year = max(float(row["annual_emissions_kg"]), 0.0)
        stacked_total = embodied_per_year + operational_per_year
        stacked_rows.append({
            "scenario": scenario_name,
            "embodied_per_year": embodied_per_year,
            "operational_per_year": operational_per_year,
            "stacked_total": stacked_total,
        })

    max_stacked_total = max([r["stacked_total"] for r in stacked_rows], default=1.0)
    if max_stacked_total <= 0:
        max_stacked_total = 1.0

    stacked_chart_rows = "".join(
        f"<div class='stacked-row'><div class='stacked-label'>{r['scenario']}</div><div class='stacked-track'><div class='stacked-segment embodied' style='width:{(r['embodied_per_year']/max_stacked_total)*100:.1f}%;'></div><div class='stacked-segment operational' style='width:{(r['operational_per_year']/max_stacked_total)*100:.1f}%;'></div></div><div class='stacked-value'>{num(r['stacked_total'])} kgCO2e/yr</div></div>"
        for r in stacked_rows
    )
    if not stacked_chart_rows:
        stacked_chart_rows = "<div style='font-size:12px;color:#666;'>No data available for stacked bar chart.</div>"

    def _safe_payback(total_value, annual_saving):
        if pd.isna(total_value) or pd.isna(annual_saving):
            return None
        if float(total_value) <= 0 or float(annual_saving) <= 0:
            return None
        return float(total_value) / float(annual_saving)

    if comparison_df.empty:
        cost_payback_chart_rows = "<div style='font-size:12px;color:#666;'>No applied renovation scenarios found.</div>"
        carbon_payback_chart_rows = "<div style='font-size:12px;color:#666;'>No applied renovation scenarios found.</div>"
    else:
        payback_df = comparison_df[["scenario", "cost_delta", "emissions_delta", "embodied_carbon_kg"]].copy()

        if available_construction_cost_col:
            construction_series = pd.to_numeric(df[available_construction_cost_col], errors="coerce")
            construction_lookup = pd.DataFrame({
                "scenario": df[scenario_col].astype(str),
                "construction_cost": construction_series,
            })
            payback_df = payback_df.merge(construction_lookup, on="scenario", how="left")
        else:
            payback_df["construction_cost"] = float("nan")

        payback_df["cost_payback_years"] = payback_df.apply(lambda r: _safe_payback(r["construction_cost"], r["cost_delta"]), axis=1)
        payback_df["carbon_payback_years"] = payback_df.apply(lambda r: _safe_payback(r["embodied_carbon_kg"], r["emissions_delta"]), axis=1)

        valid_cost_paybacks = payback_df["cost_payback_years"].dropna()
        valid_carbon_paybacks = payback_df["carbon_payback_years"].dropna()
        max_cost_payback = float(valid_cost_paybacks.max()) if not valid_cost_paybacks.empty else 1.0
        max_carbon_payback = float(valid_carbon_paybacks.max()) if not valid_carbon_paybacks.empty else 1.0

        if max_cost_payback <= 0:
            max_cost_payback = 1.0
        if max_carbon_payback <= 0:
            max_carbon_payback = 1.0

        cost_payback_chart_rows = "".join(
            f"<div class='bar-row'><div class='bar-label'>{scenario_display_map.get(str(r['scenario']), str(r['scenario']))}</div><div class='bar-track'><div class='bar retrofit' style='width:{((float(r['cost_payback_years']) / max_cost_payback) * 100.0):.1f}%;'></div></div><div class='bar-value'>{num(float(r['cost_payback_years']))} yrs</div></div>" if pd.notna(r['cost_payback_years']) else f"<div class='bar-row'><div class='bar-label'>{scenario_display_map.get(str(r['scenario']), str(r['scenario']))}</div><div class='bar-track'></div><div class='bar-value'>N/A</div></div>"
            for _, r in payback_df.iterrows()
        )

        carbon_payback_chart_rows = "".join(
            f"<div class='bar-row'><div class='bar-label'>{scenario_display_map.get(str(r['scenario']), str(r['scenario']))}</div><div class='bar-track'><div class='bar retrofit' style='width:{((float(r['carbon_payback_years']) / max_carbon_payback) * 100.0):.1f}%;'></div></div><div class='bar-value'>{num(float(r['carbon_payback_years']))} yrs</div></div>" if pd.notna(r['carbon_payback_years']) else f"<div class='bar-row'><div class='bar-label'>{scenario_display_map.get(str(r['scenario']), str(r['scenario']))}</div><div class='bar-track'></div><div class='bar-value'>N/A</div></div>"
            for _, r in payback_df.iterrows()
        )

        if not cost_payback_chart_rows:
            cost_payback_chart_rows = "<div style='font-size:12px;color:#666;'>No data available for cost payback chart.</div>"
        if not carbon_payback_chart_rows:
            carbon_payback_chart_rows = "<div style='font-size:12px;color:#666;'>No data available for carbon payback chart.</div>"

    generated_time = datetime.now().strftime("%B %d, %Y")

    html = f"""
<!DOCTYPE html>
<html lang=\"en\">
<head>
    <meta charset=\"UTF-8\">
    <meta name=\"viewport\" content=\"width=device-width, initial-scale=1.0\">
    <title>Retrofit Measure Analysis Report</title>
    <style>
        * {{ margin: 0; padding: 0; box-sizing: border-box; }}
        body {{ font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; line-height: 1.6; color: #333; background-color: #f5f5f5; }}
        .container {{ max-width: 900px; margin: 0 auto; background-color: white; padding: 40px; box-shadow: 0 0 10px rgba(0,0,0,0.1); }}
        .header {{ border-bottom: 3px solid #1f4788; padding-bottom: 20px; margin-bottom: 30px; }}
        h1 {{ color: #1f4788; font-size: 28px; margin-bottom: 5px; }}
        .subtitle {{ color: #666; font-size: 14px; margin-top: 5px; }}
        h2 {{ color: #1f4788; font-size: 18px; margin-top: 30px; margin-bottom: 15px; border-left: 4px solid #1f4788; padding-left: 10px; }}
        h3 {{ color: #1f4788; font-size: 15px; margin-top: 15px; margin-bottom: 10px; }}
        .section {{ margin-bottom: 30px; }}
        .summary-box {{ background-color: #e8f0f8; border-left: 4px solid #1f4788; padding: 15px; margin-bottom: 20px; border-radius: 3px; }}
        table {{ width: 100%; border-collapse: collapse; margin: 15px 0; }}
        th {{ background-color: #1f4788; color: white; padding: 12px; text-align: left; font-weight: bold; border: 1px solid #ddd; }}
        td {{ padding: 10px 12px; border: 1px solid #ddd; }}
        .renovation-table th:first-child,
        .renovation-table td:first-child {{ white-space: nowrap; min-width: 100px; }}
        .material-costs-table th:first-child,
        .material-costs-table td:first-child {{ white-space: nowrap; min-width: 100px; }}
        tr:nth-child(even) {{ background-color: #f9f9f9; }}
        .metric-box {{ display: grid; grid-template-columns: 1fr 1fr; gap: 15px; margin: 20px 0; }}
        .metric-card {{ background-color: #f9f9f9; border: 1px solid #ddd; padding: 15px; border-radius: 5px; text-align: center; }}
        .metric-value {{ font-size: 24px; font-weight: bold; color: #1f4788; margin: 10px 0; }}
        .metric-label {{ font-size: 12px; color: #666; text-transform: uppercase; }}
        .positive {{ color: #28a745; font-weight: bold; }}
        .viz-grid {{ display: grid; grid-template-columns: 1fr 1fr; gap: 20px; margin-top: 15px; }}
        .chart-card {{ border: 1px solid #ddd; border-radius: 6px; padding: 15px; background: #fafafa; }}
        .chart-title {{ font-size: 14px; font-weight: 600; color: #1f4788; margin-bottom: 10px; }}
        .bar-chart {{ display: grid; gap: 10px; }}
        .bar-row {{ display: grid; grid-template-columns: 130px 1fr 80px; align-items: center; gap: 10px; }}
        .bar-label {{ font-size: 12px; color: #444; }}
        .bar-track {{ height: 12px; background: #e6e6e6; border-radius: 6px; overflow: hidden; }}
        .bar {{ height: 100%; border-radius: 6px; }}
        .bar.baseline {{ background: #6c757d; }}
        .bar.retrofit {{ background: #28a745; }}
        .bar-value {{ font-size: 12px; color: #333; text-align: right; white-space: nowrap; }}
        .legend {{ display: flex; gap: 12px; margin-top: 10px; font-size: 12px; color: #555; flex-wrap: wrap; }}
        .legend-item {{ display: inline-flex; align-items: center; gap: 6px; }}
        .legend-swatch {{ width: 12px; height: 12px; border-radius: 3px; }}
        .stacked-chart {{ display: grid; gap: 10px; margin-top: 10px; }}
        .stacked-row {{ display: grid; grid-template-columns: 130px 1fr 120px; align-items: center; gap: 10px; }}
        .stacked-label {{ font-size: 12px; color: #444; }}
        .stacked-track {{ display: flex; height: 14px; background: #e6e6e6; border-radius: 7px; overflow: hidden; }}
        .stacked-segment {{ height: 100%; }}
        .stacked-segment.embodied {{ background: #fd7e14; }}
        .stacked-segment.operational {{ background: #007bff; }}
        .stacked-value {{ font-size: 12px; color: #333; text-align: right; white-space: nowrap; }}
        .iframe-wrap {{ border: 1px solid #ddd; border-radius: 6px; overflow: hidden; background: #fff; margin-top: 10px; }}
        .iframe-wrap iframe {{ width: 100%; height: 520px; border: 0; }}
        .footer {{ margin-top: 40px; padding-top: 20px; border-top: 1px solid #ddd; color: #999; font-size: 12px; text-align: center; }}
    </style>
</head>
<body>
    <div class=\"container\">
        <div class=\"header\">
            <h1>Retrofit Measure Analysis Report</h1>
            <div class=\"subtitle\">Comprehensive Energy and Financial Analysis</div>
        </div>

        <div class=\"section\">
            <h2>Executive Summary</h2>
            <div class=\"summary-box\">
                <p>This report compares energy consumption and operational costs between baseline and all applied renovation scenarios from CSV results. Embodied carbon is annualized using an analysis period of {embodied_analysis_period_years:g} years, while operational carbon is reported over {operational_analysis_period_years:g} year.</p>
            </div>
            <h3>Renovation Details by Scenario</h3>
            <table class="renovation-table">
                <tr><th>Scenario</th><th>Renovation Type</th><th>Renovation Details</th></tr>
                {renovation_rows}
            </table>
        </div>

        <div class=\"section\">
            <h2>Energy Analysis</h2>
            <table>
                <tr><th>Scenario</th><th>Metric</th><th>Baseline</th><th>Renovation</th><th>Delta</th><th>Savings %</th></tr>
                {energy_analysis_table}
            </table>
        </div>

        <div class=\"section\">
            <h2>Key Performance Metrics</h2>
            <div class=\"metric-box\">
                <div class=\"metric-card\">
                    <div class=\"metric-label\">Max Operational Cost Savings Scenario</div>
                    <div class=\"metric-value {max_cost_class}\">{money(max_cost_delta)} ({num(max_cost_delta_pct)}%)</div>
                    <div style=\"font-size:12px;color:#666;\">{max_savings_scenario}</div>
                </div>
                <div class=\"metric-card\">
                    <div class=\"metric-label\">Max Operational Emissions Reduction Scenario</div>
                    <div class=\"metric-value {max_emis_class}\">{num(max_emis_delta)} kgCO2e ({num(max_emis_delta_pct)}%)</div>
                    <div style=\"font-size:12px;color:#666;\">{max_emissions_scenario}</div>
                </div>
                <div class=\"metric-card\">
                    <div class=\"metric-label\">Min Construction Cost Scenario</div>
                    <div class=\"metric-value\">{min_construction_cost_text}</div>
                    <div style=\"font-size:12px;color:#666;\">{min_construction_cost_scenario}</div>
                </div>
                <div class=\"metric-card\">
                    <div class=\"metric-label\">Min Embodied Carbon Scenario</div>
                    <div class=\"metric-value\">{min_embodied_text}</div>
                    <div style=\"font-size:12px;color:#666;\">{min_embodied_scenario}</div>
                </div>
                <div class=\"metric-card\">
                    <div class=\"metric-label\">Baseline Energy Cost</div>
                    <div class=\"metric-value\">{money(energy_cost_per_gj)}/GJ</div>
                </div>
                <div class=\"metric-card\">
                    <div class=\"metric-label\">Scenarios With Cost Savings</div>
                    <div class=\"metric-value\">{savings_count}/{total_compared} ({num(savings_share_pct)}%)</div>
                </div>
            </div>
        </div>

        <div class=\"section\">
            <h2>Retrofit Material Costs</h2>
            <div class=\"summary-box\">
                <p>Table below shows the same scenario results visualized in the spider chart.</p>
            </div>
            <table class="material-costs-table">
                <tr><th>Scenario</th><th>Total Embodied Carbon (kgCO2e)</th><th>Annual Electricity Emissions (kgCO2e)</th><th>Annual Gas Emissions (kgCO2e)</th><th>Annual Electricity Cost (USD)</th><th>Annual Gas Cost (USD)</th></tr>
                {spider_table_rows}
            </table>
        </div>

        <div class=\"section\">
            <h2>Comparative Visualizations</h2>
            <div class=\"viz-grid\">
                <div class=\"chart-card\">
                    <div class=\"chart-title\">Annual Operational Cost (Baseline vs Max Cost Savings Scenario)</div>
                    <div class=\"bar-chart\">
                        <div class=\"bar-row\">
                            <div class=\"bar-label\">Baseline</div>
                            <div class=\"bar-track\"><div class=\"bar baseline\" style=\"width: {baseline_cost_w:.1f}%;\"></div></div>
                            <div class=\"bar-value\">{money(b['annual_cost_usd'])}</div>
                        </div>
                        <div class=\"bar-row\">
                            <div class=\"bar-label\">Max Savings</div>
                            <div class=\"bar-track\"><div class=\"bar retrofit\" style=\"width: {best_cost_w:.1f}%;\"></div></div>
                            <div class=\"bar-value\">{money(max_savings['annual_cost_usd'])}</div>
                        </div>
                    </div>
                    <div class=\"legend\">
                        <span class=\"legend-item\"><span class=\"legend-swatch\" style=\"background:#6c757d\"></span>Baseline</span>
                        <span class=\"legend-item\"><span class=\"legend-swatch\" style=\"background:#28a745\"></span>Max Operational Cost Savings Scenario</span>
                    </div>
                </div>

                <div class=\"chart-card\">
                    <div class=\"chart-title\">Operational Emissions (Baseline vs Max Reduction Scenario)</div>
                    <div class=\"bar-chart\">
                        <div class=\"bar-row\">
                            <div class=\"bar-label\">Baseline</div>
                            <div class=\"bar-track\"><div class=\"bar baseline\" style=\"width: {baseline_emis_w:.1f}%;\"></div></div>
                            <div class=\"bar-value\">{num(b['annual_emissions_kg'])}</div>
                        </div>
                        <div class=\"bar-row\">
                            <div class=\"bar-label\">Max Reduction</div>
                            <div class=\"bar-track\"><div class=\"bar retrofit\" style=\"width: {best_emis_w:.1f}%;\"></div></div>
                            <div class=\"bar-value\">{num(max_emissions_reduction['annual_emissions_kg'])}</div>
                        </div>
                    </div>
                    <div class=\"legend\">
                        <span class=\"legend-item\"><span class=\"legend-swatch\" style=\"background:#6c757d\"></span>Baseline</span>
                        <span class=\"legend-item\"><span class=\"legend-swatch\" style=\"background:#28a745\"></span>Max Operational Emissions Reduction Scenario</span>
                    </div>
                </div>
            </div>

            <div class=\"chart-card\" style=\"margin-top: 20px;\">
                <div class=\"chart-title\">Spider Chart Visualization</div>
                <p style=\"font-size:12px;color:#666;margin-bottom:8px;\">Embedded from parametric_spider_chart.html in the same output folder.</p>
                <div class=\"iframe-wrap\">
                    <iframe src=\"parametric_spider_chart.html\" title=\"Spider Chart Visualization\"></iframe>
                </div>
            </div>

            <div class=\"chart-card\" style=\"margin-top: 20px;\">
                <div class=\"chart-title\">Stacked Bar Chart Visualization</div>
                <p style=\"font-size:12px;color:#666;margin-bottom:8px;\">Orange = total embodied carbon per year (total embodied carbon / analysis period), Blue = total operational carbon per year.</p>
                <div class=\"stacked-chart\">
                    {stacked_chart_rows}
                </div>
                <div class=\"legend\">
                    <span class=\"legend-item\"><span class=\"legend-swatch\" style=\"background:#fd7e14\"></span>Total Embodied Carbon per Year</span>
                    <span class=\"legend-item\"><span class=\"legend-swatch\" style=\"background:#007bff\"></span>Total Operational Carbon per Year</span>
                </div>
            </div>
        </div>

        <div class=\"section\">
            <h2>Payback Period</h2>
            <div class=\"viz-grid\">
                <div class=\"chart-card\">
                    <div class=\"chart-title\">Cost Payback Period</div>
                    <p style=\"font-size:12px;color:#666;margin-bottom:8px;\">Payback = total construction cost / operational cost saving per year.</p>
                    <div class=\"bar-chart\">
                        {cost_payback_chart_rows}
                    </div>
                </div>

                <div class=\"chart-card\">
                    <div class=\"chart-title\">Carbon Payback Period</div>
                    <p style=\"font-size:12px;color:#666;margin-bottom:8px;\">Payback = total embodied carbon / operational carbon saving per year.</p>
                    <div class=\"bar-chart\">
                        {carbon_payback_chart_rows}
                    </div>
                </div>
            </div>
        </div>

        <div class=\"footer\">
            <p>Generated: {generated_time} | Report Type: Retrofit Impact Analysis | Run: {run_name}</p>
        </div>
    </div>
</body>
</html>
"""

    html_report_path.write_text(html, encoding="utf-8")
    return html_report_path


base_dir = Path.cwd() / "simulations" / RUN_NAME
csv_candidates = [
    base_dir / "parameter_results.csv",
    base_dir / "parametric_results.csv",
]
csv_path = next((p for p in csv_candidates if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError(f"Could not find CSV in: {csv_candidates}")

df_report = pd.read_csv(csv_path)
html_output_path = base_dir / "parametric_report.html"
out = generate_html_report(df_report, html_output_path, run_name=RUN_NAME)
print(f"HTML report generated: {out}")

HTML report generated: c:\All repos\openstudio-ee-gem\lib\parametric_run\simulations\run_test_001\parametric_report.html
